In [2]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt


In [5]:
data = pd.read_csv('C:/Users/User/Desktop/DATA LEARN/Amdari/Data Analysis/week 1/Customer-Analytics-and-Churn-Prediction/Customer-Analytics-and-Churn-Prediction-1/data/drivers.csv')

In [17]:
data_files= {
    'drivers':('signup_date','last_active'),
    'riders' :('signup_date',),
    'trips' :('pickup_time','dropoff_time'),
    'sessions'  : ('session_time',),
    'promotions': ('start_date','end_date')
}

dfs={}
for name, date_cols in data_files.items():
    dfs[name]= pd.read_csv(f'../data/{name}.csv', parse_dates = list(date_cols))

In [20]:
dfs['sessions']

,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 18:57:06+02:05,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 07:32:22+02:27,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 23:17:25+02:05,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:40:25+00:14,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:31:22+00:14,4,1,0,Lagos,Bronze
...,...,...,...,...,...,...,...,...
49995,S049995,R03092,2025-04-27 07:52:18+00:14,76,5,0,Lagos,Bronze
49996,S049996,R02272,2025-04-27 16:52:12+02:05,44,1,1,Cairo,Bronze
49997,S049997,R06892,2025-04-27 14:02:26+02:27,9,5,0,Nairobi,Bronze
49998,S049998,R03121,2025-04-27 03:45:34+00:14,65,2,0,Lagos,Bronze


### Task one
- check the hypothesis on the converted riders are stored in trips
- check whether riders table contains information where sessions converted is 1

### classification problem

- Engineer our target => we set a threshold to determine churn or not churn where not churn is 0 => 0.5 and churn is 1

In [21]:
dfs['riders']

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,NaN
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,NaN
3,R00003,2025-03-17,Bronze,41.658628,Nairobi,4.9,0.244779,NaN
4,R00004,2024-08-20,Silver,40.681709,Lagos,3.9,0.269960,R00002
...,...,...,...,...,...,...,...,...
9995,R09995,2025-04-26,Silver,20.688176,Lagos,4.4,0.028371,NaN
9996,R09996,2023-07-26,Silver,18.000000,Cairo,4.1,0.290365,NaN
9997,R09997,2025-04-18,Bronze,34.197784,Lagos,4.7,0.013147,R02155
9998,R09998,2024-01-10,Bronze,51.705319,Cairo,3.8,0.253959,R07447


In [22]:
dfs['trips']

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 05:36:41+02:27,2025-02-17 05:52:41+02:27,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 19:27:14+02:05,2024-06-18 19:32:14+02:05,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 09:58:16+02:27,2024-10-05 10:28:16+02:27,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,T199995,R08022,D04562,26.79,1.3,0.00,Card,2025-03-14 18:16:26+00:14,2025-03-14 18:27:26+00:14,6.511389,3.297189,6.545565,3.327685,Cloudy,Lagos,Silver
199996,T199996,R05421,D03984,14.65,1.0,0.00,Card,2024-07-02 06:59:36+00:14,2024-07-02 07:36:36+00:14,6.489143,3.492786,6.453581,3.514289,Sunny,Lagos,Bronze
199997,T199997,R06619,D01173,12.87,1.2,0.00,Mobile Money,2024-05-06 20:38:57+00:14,2024-05-06 21:25:57+00:14,6.459348,3.527623,6.451930,3.521616,Rainy,Lagos,Bronze
199998,T199998,R02867,D00974,17.18,1.3,0.00,Mobile Money,2024-09-25 03:11:33+00:14,2024-09-25 03:45:33+00:14,6.540074,3.471683,6.540339,3.426481,Rainy,Lagos,Gold


In [23]:
dfs['promotions']

,promo_id,promo_name,promo_type,promo_value,start_date,end_date,target_segment,city_scope,ab_test_groups,test_allocation,success_metric
0,P000,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-25,All,Nairobi,['All'],[1.0],Usage Frequency
1,P001,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-22,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate
2,P002,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-16,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",ROI
3,P003,Loyalty Bonus,points,100.0,2025-04-26,2025-05-04,Gold+,Nairobi,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate
4,P004,Loyalty Bonus,points,100.0,2025-04-26,2025-05-15,Gold+,Nairobi,['All'],[1.0],Usage Frequency
5,P005,Loyalty Bonus,points,100.0,2025-04-26,2025-05-16,Gold+,Nairobi,['All'],[1.0],Usage Frequency
6,P006,First Ride Discount,discount,0.3,2025-04-26,2025-05-06,New,Lagos,['All'],[1.0],Usage Frequency
7,P007,First Ride Discount,discount,0.3,2025-04-26,2025-05-26,New,Nairobi,['All'],[1.0],ROI
8,P008,Referral Special,credit,15.0,2025-04-26,2025-05-03,Active,All-Cities,['All'],[1.0],Usage Frequency
9,P009,Referral Special,credit,15.0,2025-04-26,2025-05-18,Active,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",ROI


### Basic Data Quality
- null values
- duplicates
- overall information using .info()

In [36]:
dfs['drivers'].duplicated().any()

np.False_

In [38]:
dfs['drivers'].isnull().any()

driver_id          False
rating             False
vehicle_type       False
signup_date        False
last_active        False
city               False
acceptance_rate    False
dtype: bool

In [25]:
dfs['riders'].isnull().any()

user_id             False
signup_date         False
loyalty_status      False
age                 False
city                False
avg_rating_given    False
churn_prob          False
referred_by          True
dtype: bool

In [32]:
dfs['riders'].duplicated().any()

np.False_

In [26]:
dfs['trips'].isnull().any()

trip_id             False
user_id             False
driver_id           False
fare                False
surge_multiplier    False
tip                 False
payment_type        False
pickup_time         False
dropoff_time        False
pickup_lat          False
pickup_lng          False
dropoff_lat         False
dropoff_lng         False
weather             False
city                False
loyalty_status      False
dtype: bool

In [33]:
dfs['trips'].duplicated().any()

np.False_

In [27]:
dfs['sessions'].isnull().any()

session_id        False
rider_id          False
session_time      False
time_on_app       False
pages_visited     False
converted         False
city              False
loyalty_status    False
dtype: bool

In [34]:
dfs['sessions'].duplicated().any()

np.False_

In [29]:
dfs['promotions'].isnull().any()

promo_id           False
promo_name         False
promo_type         False
promo_value        False
start_date         False
end_date           False
target_segment     False
city_scope         False
ab_test_groups     False
test_allocation    False
success_metric     False
dtype: bool

In [35]:
dfs['promotions'].duplicated().any()

np.False_

np.int64(0)

In [51]:

for name in data_files.keys():
    print('='*15+' '+ name +' '+ '='*15)
    print(dfs[name].info())
    print('*'*40)
    



=============== drivers ===============
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   driver_id        5000 non-null   str           
 1   rating           5000 non-null   float64       
 2   vehicle_type     5000 non-null   str           
 3   signup_date      5000 non-null   datetime64[us]
 4   last_active      5000 non-null   datetime64[us]
 5   city             5000 non-null   str           
 6   acceptance_rate  5000 non-null   float64       
dtypes: datetime64[us](2), float64(2), str(3)
memory usage: 273.6 KB
None
****************************************
=============== riders ===============
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           10000 n

In [55]:
drivers = dfs['drivers']
promotions = dfs['promotions']
riders = dfs['riders']
sessions= dfs['sessions']
trips = dfs['trips']


In [ ]:

trips['pickup_time'] = pd.to_datetime(trips['pickup_time'], utc=True )
trips['dropoff_time'] = pd.to_datetime(trips['dropoff_time'], utc=True )
sessions['session_time'] = pd.to_datetime(trips['session_time'], utc=True )


ValueError: Length of values (1) does not match length of index (200000)

In [69]:
sessions


,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 18:57:06+02:05,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 07:32:22+02:27,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 23:17:25+02:05,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:40:25+00:14,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:31:22+00:14,4,1,0,Lagos,Bronze
...,...,...,...,...,...,...,...,...
49995,S049995,R03092,2025-04-27 07:52:18+00:14,76,5,0,Lagos,Bronze
49996,S049996,R02272,2025-04-27 16:52:12+02:05,44,1,1,Cairo,Bronze
49997,S049997,R06892,2025-04-27 14:02:26+02:27,9,5,0,Nairobi,Bronze
49998,S049998,R03121,2025-04-27 03:45:34+00:14,65,2,0,Lagos,Bronze
